# EDA Notebook for Keithley CLI data

This notebook is used for exploratory data analysis (EDA) of the measurement data obtained from the Keithley CLI.

In [134]:
## Importing Libraries and Helper Functions
import plotly.express as px
import pandas as pd
from helpers.nextcloud import nextcloud_txt_to_dataframe, nextcloud_csv_to_dataframe

In [135]:
## Download and combine .txt files into a DataFrame, then print row counts per source file.
df = nextcloud_txt_to_dataframe()

# Ignore some of the file containing certain keywords, and plot the rest. For example, if the file name contains "_vdramp", we can assume it's Sternberg data and plot it separately.
ignore_keywords = [
    "dummy",
    "friday",
    "monday_pre",
    "empty",
]  # Add more keywords as needed

df = df[
    ~df["source_file"].str.contains("|".join(ignore_keywords), case=False, na=False)
]

df["source_file"] = (
    df["source_file"].str.split("/").str[-1]
)  # Keep only the filename, not the path

df["abs(Gate Current)"] = df["Gate Current"].abs()
df["abs(Drain Current)"] = df["Drain Current"].abs()

# Calculate flux as the change in fluence over the change in time.
df["Flux"] = df["Fluence"].diff() / df["Time"].diff()
# Interpolate any single zero value in Flux by taking the average of the previous and next values. This is to handle any potential division by zero issues when calculating flux.
zero_flux_indices = df[df["Flux"] == 0].index
for idx in zero_flux_indices:
    if idx > 0 and idx < len(df) - 1:  # Ensure we have previous and next values
        df.at[idx, "Flux"] = (df.at[idx - 1, "Flux"] + df.at[idx + 1, "Flux"]) / 2


print("RADEF txt files:")
for file in df["source_file"].unique() if not df.empty else []:
    subset = df[df["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

if df.empty:
    print("No data loaded from the Nextcloud share.")


RADEF txt files:
File: a31_000_gq.txt, Data Rows: 41
File: a31_000_idvd.txt, Data Rows: 111
File: a31_000_idvg.txt, Data Rows: 201
File: a31_001_vdramp.txt, Data Rows: 175
File: a32_000_gq.txt, Data Rows: 41
File: a32_000_idvd.txt, Data Rows: 111
File: a32_000_idvg.txt, Data Rows: 201
File: a32_001_gq.txt, Data Rows: 461
File: a32_001_idvd.txt, Data Rows: 111
File: a32_001_idvg.txt, Data Rows: 201
File: a32_001_irrad.txt, Data Rows: 254
File: a32_001b_gq.txt, Data Rows: 205
File: a32_002_irrad.txt, Data Rows: 173
File: a33_000_gq.txt, Data Rows: 40
File: a33_000_idvd.txt, Data Rows: 91
File: a33_000_idvg.txt, Data Rows: 201
File: a33_001_vdramp.txt, Data Rows: 195
File: a34_000_gq.txt, Data Rows: 38
File: a34_000_idvd.txt, Data Rows: 91
File: a34_000_idvg.txt, Data Rows: 201
File: a34_001_gq.txt, Data Rows: 458
File: a34_001_idvd.txt, Data Rows: 91
File: a34_001_idvg.txt, Data Rows: 201
File: a34_001_irrad.txt, Data Rows: 224
File: a34_001b_gq.txt, Data Rows: 192
File: a34_002_irrad.tx

## Gate Charge Data

In [136]:
# Get gate charge data and plot gate voltage vs. gate charge for files containing "_gq" in their name.

gate_charge_df = df[df["source_file"].str.contains("_gq", case=False, na=False)]

if gate_charge_df.empty:
    print("No gate charge data found in the DataFrame.")
else:
    # Integrate gate current over time to calculate gate charge (Q = \int_0^t I(t) * dt).
    dt = gate_charge_df["Time"].diff().fillna(0)  # Time intervals
    gate_charge_df["Gate Charge"] = (
        gate_charge_df["Gate Current"] * dt
    ).cumsum()  # Cumulative sum to get total charge

    fig = px.line(
        gate_charge_df, x="Gate Charge", y="Gate Voltage", color="source_file"
    )

    fig.update_layout(
        width=800,
        height=600,
        xaxis_range=[0, 50e-9],
    )
    fig.show()

## ID-VD Data

In [137]:
# Get ID-VD data and plot Drain Current and Gate Current vs. Drain Voltage on the same graph, with separate lines for each source file.
idvd_df = df[df["source_file"].str.contains("_idvd", case=False, na=False)]
if idvd_df.empty:
    print("No ID-VD data found in the DataFrame.")
else:
    fig_idvd = px.line(
        idvd_df,
        x="Drain Voltage",
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        symbol="variable",
    )
    fig_idvd.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvd.show()

## ID-VG Data

In [138]:
# Get ID-VG data and plot Drain Current and Gate Current vs. Gate Voltage on the same graph, with separate lines for each source file.
idvg_df = df[df["source_file"].str.contains("_idvg", case=False, na=False)]

if idvg_df.empty:
    print("No ID-VG data found in the DataFrame.")
else:
    fig_idvg = px.line(
        idvg_df,
        x="Gate Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        markers=True,
        symbol="variable",
    )

    fig_idvg.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvg.show()


## Sternberg Data

In [139]:
## Sternberg Data

# Ignore some of the file containing certain keywords, and plot the rest. For example, if the file name contains "_vdramp", we can assume it's Sternberg data and plot it separately.
ignore_keywords = ["dummy", "friday"]  # Add more keywords as needed

sternberg_df = df[df["source_file"].str.contains("_vdramp", case=False, na=False)]

x_col = "Time"
# sort by column x_col
sternberg_df = sternberg_df.sort_values(by=x_col)

if sternberg_df.empty:
    print("No Sternberg data found in the DataFrame.")
else:
    fig_sternberg = px.line(
        sternberg_df,
        x=x_col,
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        markers=True,
        symbol="variable",
        hover_data=["Time", "Drain Voltage", "Fluence", "Flux"],
    )

    fig_sternberg.update_layout(
        width=800,
        height=600,
        yaxis_type="linear",  # Logarithmic scale for current to better visualize low currents
    )
    fig_sternberg.show()

In [ ]:
# Isolate leakage-current stepping runs when beam is ON and Vd is around 250 V or 350 V.

# dataset = "a47_001_vdramp"
dataset = "a49_001_vdramp"

min_points_per_run = 60

mapping_df = sternberg_df[sternberg_df["source_file"].str.contains(dataset)].copy()
mapping_df = mapping_df.sort_values("Time").reset_index(drop=True)

# Normalize Flux to a boolean beam-on mask.
flux_num = pd.to_numeric(mapping_df["Flux"], errors="coerce")
beam_on_numeric = flux_num.gt(1)
beam_on = beam_on_numeric

# Group drain bias into rough mapping bins around 250 V and 350 V.
vdrain_abs = pd.to_numeric(mapping_df["Drain Voltage"], errors="coerce").abs()
mapping_df["bias_group"] = pd.NA
mapping_df.loc[vdrain_abs.between(235, 265), "bias_group"] = 250
mapping_df.loc[vdrain_abs.between(335, 365), "bias_group"] = 350

active = beam_on & mapping_df["bias_group"].notna()

# Start a new run when entering an active region or when the active bias group changes.
new_run = active & (
    (~active.shift(fill_value=False))
    | (mapping_df["bias_group"] != mapping_df["bias_group"].shift())
)

mapping_df["run_id"] = new_run.cumsum().where(active).astype("Int64")
mapping_df["bias_group"] = mapping_df["bias_group"].astype("Int64")

# Keep only runs that have at least min_points_per_run samples.
run_sizes = mapping_df[mapping_df["run_id"].notna()].groupby("run_id").size()

valid_run_ids = run_sizes[run_sizes >= min_points_per_run].index

df_runs = mapping_df[mapping_df["run_id"].isin(valid_run_ids)].copy()
df_runs["Run Time"] = df_runs.groupby("run_id")["Time"].transform(
    lambda x: x - x.iloc[0]
)
df_runs["Run Fluence"] = df_runs.groupby("run_id")["Fluence"].transform(
    lambda x: x - x.iloc[0]
)
df_runs["Run Delta ID"] = df_runs.groupby("run_id")["Drain Current"].transform(
    lambda x: x - x.iloc[0]
)


if df_runs.empty:
    print(
        f"No mapping runs found with at least {min_points_per_run} points "
        "for beam ON and Vd around 250/350 V."
    )
else:
    # Re-index surviving runs to 1..N in chronological order.
    run_id_map = {old_id: i + 1 for i, old_id in enumerate(sorted(valid_run_ids))}
    df_runs["run_id"] = df_runs["run_id"].map(run_id_map).astype("Int64")
    df_runs["run_label"] = (
        "Vd~"
        + df_runs["bias_group"].astype(str)
        + "_run"
        + df_runs["run_id"].astype(str)
    )

    print(df_runs.groupby(["run_id", "bias_group"]).size().reset_index(name="n_points"))

    fig = px.line(
        df_runs,
        x="Run Fluence",
        y=["Run Delta ID"],
        color="run_label",
        markers=True,
        # symbol="variable",
        hover_data=["Time", "Drain Voltage", "Fluence", "Flux", "run_id", "bias_group"],
    )

    fig.update_layout(
        width=900,
        height=600,
        yaxis_type="linear",
        xaxis_range=[0, df_runs["Run Fluence"].max() * 1.05],
    )
    fig.show()


   run_id  bias_group  n_points
0       1         250       309
1       2         350       344
2       3         250       341
3       4         350       358
4       5         250       328
5       6         350       348


In [141]:
df_runs

,Time,Gate Voltage,Gate Current,Drain Voltage,Drain Current,source_file,Fluence,abs(Gate Current),abs(Drain Current),Flux,bias_group,run_id,Run Time,Run Fluence,Run Delta ID,run_label
117,165.761222,0.000011,-5.646110e-13,252.7468,-1.119230e-09,a47_001_vdramp.txt,12.0,5.646110e-13,1.119230e-09,3.200321,250,1,0.000000,0.0,0.000000e+00,Vd~250_run1
118,167.167331,0.000004,-8.438890e-12,252.7473,-1.754803e-09,a47_001_vdramp.txt,21.0,8.438890e-12,1.754803e-09,6.400642,250,1,1.406109,9.0,-6.355730e-10,Vd~250_run1
119,168.612030,0.000011,-1.861010e-08,252.7467,1.761168e-08,a47_001_vdramp.txt,364.0,1.861010e-08,1.761168e-08,237.419698,250,1,2.850808,352.0,1.873091e-08,Vd~250_run1
120,171.106070,0.000010,-1.889270e-08,252.7452,1.735918e-08,a47_001_vdramp.txt,407.0,1.889270e-08,1.735918e-08,17.241103,250,1,5.344848,395.0,1.847841e-08,Vd~250_run1
121,171.316033,0.000011,-1.868480e-08,252.7475,1.743412e-08,a47_001_vdramp.txt,435.0,1.868480e-08,1.743412e-08,133.356830,250,1,5.554811,423.0,1.855335e-08,Vd~250_run1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3313,813.333855,0.000010,-1.993520e-06,356.7029,2.005237e-06,a47_001_vdramp.txt,60036.0,1.993520e-06,2.005237e-06,159.751818,350,6,67.913502,9982.0,7.288140e-07,Vd~350_run6
3314,813.526744,0.000010,-2.018410e-06,358.5980,2.026532e-06,a47_001_vdramp.txt,60067.0,2.018410e-06,2.026532e-06,160.714193,350,6,68.106391,10013.0,7.501090e-07,Vd~350_run6
3315,813.732444,0.000011,-2.042680e-06,360.6408,2.050467e-06,a47_001_vdramp.txt,60098.0,2.042680e-06,2.050467e-06,150.704910,350,6,68.312091,10044.0,7.740440e-07,Vd~350_run6
3316,813.926131,0.000008,-2.062900e-06,362.5919,2.067347e-06,a47_001_vdramp.txt,60129.0,2.062900e-06,2.067347e-06,160.052043,350,6,68.505778,10075.0,7.909240e-07,Vd~350_run6


## Irradiation Data

In [142]:
# Get irradiation data and plot Drain Current and Gate Current vs. Fluence on the same graph, with separate lines for each source file.
irrad_df = df[df["source_file"].str.contains("_irrad", case=False, na=False)]

if irrad_df.empty:
    print("No irradiation data found in the DataFrame.")
else:
    fig_irrad = px.line(
        irrad_df,
        x="Fluence",
        y=["abs(Drain Current)", "abs(Gate Current)"],
        color="source_file",
        markers=True,
        symbol="variable",
        title="Irradiation Test Data",
    )

    fig_irrad.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_irrad.show()

## Sanity Check Between Vanderbilt and JYU Data

In [143]:
# Get Joshua's CSV files and plot them alongside RADEF's data for comparison.
df_joshua = nextcloud_csv_to_dataframe()

print("Joshua's CSV files:")
for file in df_joshua["source_file"].unique() if not df_joshua.empty else []:
    subset = df_joshua[df_joshua["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

idvg_df_joshua = df_joshua[
    df_joshua["source_file"].str.contains("_idvg", case=False, na=False)
]
# Remove DataName column if it exists, as it may interfere with plotting
idvg_df_joshua = idvg_df_joshua.drop(columns=["DataName"], errors="ignore")

# Rename joshua's ID-VG columns to match RADEF's for easier plotting
col_mapping = {
    "Vgate": "Gate Voltage",
    "Isource": "Source Voltage",
    "Idrain": "Drain Current",
    "Igate": "Gate Current",
}

# strip whitespace from column names in joshua's DataFrame before renaming
idvg_df_joshua.columns = idvg_df_joshua.columns.str.strip()

idvg_df_joshua = idvg_df_joshua.rename(columns=col_mapping)

# Combine both dataframes
idvg_combined = pd.concat([idvg_df, idvg_df_joshua], ignore_index=True)

fig_combined = px.line(
    idvg_combined,
    x="Gate Voltage",
    y=["Drain Current", "Gate Current"],
    color="source_file",
    symbol="variable",
    title="Combined ID-VG Characteristics (RADEF and Joshua's Data)",
)

fig_combined.update_layout(
    width=800,
    height=600,
    yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
)
fig_combined.show()


Joshua's CSV files:
File: raw_data/friday/A010_3_4_2026 9_19_32 AM_idvg.csv, Data Rows: 126
